In [1]:
%reload_ext autoreload
%autoreload 2

In [83]:
from tqdm.auto import tqdm
import pandas as pd
from typing import Optional, List, Tuple
import json
import datasets
import pprint
pd.set_option("display.max_colwidth", None)

In [30]:
from huggingface_hub import notebook_login

notebook_login()

### Load dataset knowledge base

In [4]:
ds = datasets.load_dataset("m-ric/huggingface_doc", split="train")

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv: reconstructing file:   0%|          |  0.00B / 22.0MB            

huggingface_doc.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

In [ ]:
ds["text"]

Column([' Create an Endpoint\n\nAfter your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. \n\n## 1. Enter the Hugging Face Repository ID and your desired endpoint name:\n\n<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_repository.png" alt="select repository" />\n\n## 2. Select your Cloud Provider and region. Initially, only AWS will be available as a Cloud Provider with the `us-east-1` and `eu-west-1` regions. We will add Azure soon, and if you need to test Endpoints with other Cloud Providers or regions, please let us know.\n\n<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_region.png" alt="select region" />\n\n## 3. Defin

In [9]:
ds

Dataset({
    features: ['text', 'source'],
    num_rows: 2647
})

In [13]:
len(ds["text"])

2647

In [11]:
ds[0]

{'text': ' Create an Endpoint\n\nAfter your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. \n\n## 1. Enter the Hugging Face Repository ID and your desired endpoint name:\n\n<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_repository.png" alt="select repository" />\n\n## 2. Select your Cloud Provider and region. Initially, only AWS will be available as a Cloud Provider with the `us-east-1` and `eu-west-1` regions. We will add Azure soon, and if you need to test Endpoints with other Cloud Providers or regions, please let us know.\n\n<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_region.png" alt="select region" />\n\n## 3. Defi

### Build a synthetic dataset for evaluation

In [31]:
# prepare source documents
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LangchainDocument

langchain_docs = [LangchainDocument(page_content=doc["text"], metadata={"source": doc["source"]}) for doc in tqdm(ds)]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    add_start_index=True,
    separators=["\n\n", "\n", ".", " ", ""]
)

docs_processed = []
for doc in langchain_docs:
    docs_processed += text_splitter.split_documents([doc])

  0%|          | 0/2647 [00:00<?, ?it/s]

In [78]:
docs_processed[0]

Document(metadata={'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx', 'start_index': 1}, page_content='Create an Endpoint\n\nAfter your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. \n\n## 1. Enter the Hugging Face Repository ID and your desired endpoint name:\n\n<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_repository.png" alt="select repository" />\n\n## 2. Select your Cloud Provider and region. Initially, only AWS will be available as a Cloud Provider with the `us-east-1` and `eu-west-1` regions. We will add Azure soon, and if you need to test Endpoints with other Cloud Providers or regions, please let us know.\n\n<im

In [ ]:
hf_token = ""

In [151]:
from huggingface_hub import InferenceClient
repo_id = "Qwen/Qwen3-4B-Thinking-2507"
# repo_id="meta-llama/Llama-3.2-8B-Instruct"
# repo_id="Qwen/Qwen2.5-0.5B-Instruct"

llm_client = InferenceClient(model=repo_id, token=hf_token)

def call_llm(inference_client: InferenceClient, prompt: str):

    response = inference_client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2048
    )

    return response.choices[0].message.content

output = call_llm(llm_client, "This is a text context prompt.")

In [130]:
output

"I'm ready to respond. What's the text you'd like me to respond to?"

In [131]:
QA_generation_prompt = """
Your task is to write a factoid question and an answer given a context.
Your factoid question should be answerable with a specific, concise piece of factual information from the context.
Your factoid question should be formulated in the same style as questions users could ask in a search engine.
This means that your factoid question MUST NOT mention something like "according to the passage" or "context".

Provide your answer as follows:

Output:::
Factoid question: (your factoid question)
Answer: (your answer to the factoid question)

Now here is the context.

Context: {context}\n
Output:::
"""

In [152]:
# generate QA couples
import random

N_GENERATIONS = 10
print(f"Generating {N_GENERATIONS} QA couples...")

outputs = []
raw_outputs = []

for sampled_context in tqdm(random.sample(docs_processed, N_GENERATIONS)):
    # Generate QA couples
    output_QA_couple = call_llm(llm_client, QA_generation_prompt.format(context=sampled_context.page_content))
    try:

        raw_outputs.append(output_QA_couple)
        question = output_QA_couple.split("Factoid question: ")[-1].split("Answer: ")[0]
        answer = output_QA_couple.split("Answer: ")[-1]
        outputs.append(
            {
                "context": sampled_context.page_content,
                "question": question,
                "answer": answer,
                "source_doc": sampled_context.metadata["source"],
            }
        )
    except:
        continue

    

Generating 10 QA couples...


  0%|          | 0/10 [00:00<?, ?it/s]

HfHubHTTPError: Server error '500 Internal Server Error' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6abc3464-66f63f8012a8f9e530973f8a;263d5495-711a-4c8f-8e8f-147b854ea492)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500

{'code': '500', 'error_type': 'INTERNAL_ERROR', 'message': 'Error from model provider. Contact support if this persists.', 'param': None}

In [ ]:
pprint.pprint(print(outputs[0]))

We are given a context about integrations of the HfFileSystem with various libraries.
 The task: write a factoid question that is answerable with a specific, concise fact from the context, and the question should be in the style of a search engine query (no "according to the passage" etc.)

 Let's look for specific, concise facts in the context that could form a question.

 Examples of facts in the context:
 1. The URL scheme for the HfFileSystem: `hf://[<repo_type_prefix>]<repo_id>[@<revision>]/<path/in/repo>`
 2. The `repo_type_prefix` values: 
      - `datasets/` for datasets
      - `spaces/` for spaces
      - models don't need a prefix
 3. Examples of integrations: 
      - Reading/writing a Pandas DataFrame
      - Dask, Polars
      - DuckDB for querying remote Hub files
      - Zarr for array stores

 We need to form a question that is specific and answerable by a single fact.

 Let's try to pick one fact that is both specific and can be phrased as a question without the conte

In [ ]:
pprint.pprint(print(outputs[0].split("Answer: ")[-1])) 

"hf://[<repo_type
None


In [ ]:
outputs[1].split("Factoid question: ")[-1].split("Answer: ")[0]

"What is Adversarial Inception v3's top 1 accuracy on ImageNet?\n"